<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeshengliu213/ai-for-materials/blob/main/04%20-%20Bayesian%20Optimization/04b_bayesian_optimization.ipynb)

# 04b · 贝叶斯优化（下）：用最少的实验找到最优材料配方

> 对应林雪平大学（LiU）`Lecture 08 - Bayesian learning` 下半部分，并融合真实钯催化 Buchwald-Hartwig 偶联反应数据集（Ahneman et al., *Science* 2018）。

上一节（`04a`）我们学会了用高斯过程回归（GPR）同时预测均值 $\mu(x)$ 和不确定性 $\sigma(x)$。今天解决真正让实验材料人头疼的问题：
**假设我们有 4000 种可能的反应配方（或者一片温度/浓度二维参数空间），每次实验室合成都要花 1 天。我们不能把 4000 个全试一遍，如何用 20 次实验就找到产率最高的那组配方？**

这就是 **贝叶斯优化（Bayesian Optimization, BO）**——也是如今"自动驾驶实验室（Self-Driving Labs）"的大脑核心。

1. 贝叶斯优化的闭环工作流
2. 三大采集函数：Greedy、UCB、EI（手写实现）
3. 1D 材料工艺寻优实战：一步步看 BO 如何避开局部最优陷阱
4. 真实反应寻优：Buchwald-Hartwig 钯催化偶联反应（3955 个配方里 20 次找到 >90% 产率）
5. 贝叶斯优化的边界与使用忠告

### 这个 notebook 怎么用

| 标记 | 意思 | 你要做什么 |
|---|---|---|
| 📘 | 讲解 | 读一读 |
| 🤔 | 先猜一猜 | **运行之前**先想想会输出什么 |
| ▶️ | 运行看看 | 选中代码格，按 `Shift + Enter` |
| ✏️ | 改一改 | 改代码里标了 `# ✏️` 的地方，再运行 |
| 🏋️ | 练习 | 自己写代码；参考答案在 `solutions/` 文件夹 |
| 💡 | 小贴士 | 有用但不是必须记住的东西 |

In [ ]:
# colab-setup (本地运行时自动跳过)
import os, sys

if "google.colab" in sys.modules:
    _repo_dir = "/content/ai-for-materials"
    if not os.path.exists(_repo_dir):
        os.system("git clone --depth 1 https://github.com/Zeshengliu213/ai-for-materials.git " + _repo_dir)
    os.chdir(os.path.join(_repo_dir, "04 - Bayesian Optimization"))
    get_ipython().run_line_magic("pip", "install -q rdkit")


In [ ]:
try:
    import rdkit
except ImportError:
    %pip install -q rdkit
    import rdkit

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, RBF, WhiteKernel, ConstantKernel as C
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem

RDLogger.DisableLog("rdApp.*")
plt.rcParams["figure.dpi"] = 110

In [ ]:
# 本地运行：文件在 data/ 文件夹里
# Colab 运行：如果找不到文件，会弹出上传按钮（本节需要 bh-reactions.csv）
def find_file(name):
    for path in (f"data/{name}", name):
        if os.path.exists(path):
            return path
    try:
        from google.colab import files
        print(f"请上传 {name}")
        files.upload()
        return name
    except ImportError:
        raise FileNotFoundError(f"找不到 {name}，请把它放在 data/ 文件夹里")

---
## 1. 贝叶斯优化的闭环：探索（Exploration）vs 利用（Exploitation）

📘 贝叶斯优化只有短短四步循环：
1. **初始化**：先随机做少量（例如 3~5 个）实验，拿到初始数据 $(X, y)$；
2. **建代理模型**：用高斯过程（GPR）拟合现有数据，得到每个候选点的均值 $\mu(x)$ 和不确定性 $\sigma(x)$；
3. **算采集函数（Acquisition Function）$\alpha(x)$**：综合 $\mu(x)$ 和 $\sigma(x)$ 给每个还没测过的候选点打分，选分数最高的那个点去做下一轮真实实验；
4. **更新数据**：把新测出来的实验结果放回训练集，回到第 2 步，直到预算用完或性能达标。

🤔 **核心权衡：**
- **利用（Exploitation）**：去均值 $\mu(x)$ 已经很高的地方继续微调（"趁热打铁"）；
- **探索（Exploration）**：去不确定性 $\sigma(x)$ 很大的空白处看一看（"万一那里藏着更好的峰呢？"）。

---
## 2. 手写三大采集函数（Greedy / UCB / EI）

📘 最常用的三种打分规则（假设我们要**最大化**目标性能 $y$）：
1. **Greedy（纯利用）**：$\alpha_{\text{Greedy}}(x) = \mu(x)$。不管不确定性，只去预测均值最高的地方。
2. **UCB（Upper Confidence Bound，置信上界）**：
   $$\alpha_{\text{UCB}}(x) = \mu(x) + \kappa \cdot \sigma(x)$$
   其中 $\kappa \ge 0$ 由你控制。$\kappa = 0$ 就是纯利用；$\kappa = 2$ 鼓励积极探索未知区域。
3. **EI（Expected Improvement，期望提升）**：
   设当前做过的实验里最高性能是 $y_{\text{best}}$。如果模型预测某点分布为 $\mathcal{N}(\mu, \sigma^2)$，它能超越 $y_{\text{best}}$ 的**平均期望增量**为：
   $$\alpha_{\text{EI}}(x) = (\mu - y_{\text{best}} - \xi) \Phi(Z) + \sigma \phi(Z), \quad Z = \frac{\mu - y_{\text{best}} - \xi}{\sigma}$$
   其中 $\Phi$ 和 $\phi$ 分别是标准正态分布的 CDF 和 PDF，$\xi$（通常取 `0.01`）防止陷入微小改进。

▶️ 下面先定义一个"未知的真实工艺曲线"（一个双峰函数：左边有个小山头，右边有个更高的主峰），看看这三种采集函数在一个已有的 GPR 状态上会推荐去哪测下一个点。

In [ ]:
def hidden_process(x):
    # 模拟具有两个局部峰的工艺参数曲线（例如退火温度 0~10 对应器件性能）
    return 0.6 * np.exp(-0.5 * ((x - 2.3) / 0.8)**2) + 1.0 * np.exp(-0.5 * ((x - 7.8) / 0.9)**2)

def acq_ucb(mu, std, kappa=2.0):
    return mu + kappa * std

def acq_ei(mu, std, y_best, xi=0.01):
    with np.errstate(divide="ignore", invalid="ignore"):
        imp = mu - y_best - xi
        Z = np.where(std > 1e-9, imp / std, 0.0)
        ei = np.where(std > 1e-9, imp * norm.cdf(Z) + std * norm.pdf(Z), 0.0)
    return ei

X_grid = np.linspace(0, 10, 300).reshape(-1, 1)
# 假设一开始只做了 3 个实验，都在左边小山头附近
X_init = np.array([[1.2], [2.2], [3.6]])
y_init = hidden_process(X_init).ravel()

gp = GaussianProcessRegressor(kernel=C(1.0) * RBF(1.0), alpha=1e-5, random_state=0)
gp.fit(X_init, y_init)
mu, std = gp.predict(X_grid, return_std=True)

fig, axes = plt.subplots(2, 1, figsize=(9.5, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1.2]})
axes[0].plot(X_grid, hidden_process(X_grid), "k--", alpha=0.5, label="Hidden true process")
axes[0].plot(X_grid, mu, "b-", lw=2, label="GP mean")
axes[0].fill_between(X_grid.ravel(), mu - 1.96 * std, mu + 1.96 * std, color="blue", alpha=0.15, label="95% CI")
axes[0].scatter(X_init, y_init, c="red", s=60, zorder=5, label="Observed (n=3)")
axes[0].set_ylabel("Device Performance")
axes[0].set_title("Surrogate Model vs Acquisition Functions")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

ei_vals = acq_ei(mu, std, y_init.max(), xi=0.01)
ucb_vals = acq_ucb(mu, std, kappa=2.0)

axes[1].plot(X_grid, ei_vals / (ei_vals.max() + 1e-9), "g-", lw=2, label=f"EI (next x={X_grid[np.argmax(ei_vals),0]:.2f})")
axes[1].plot(X_grid, ucb_vals / (ucb_vals.max() + 1e-9), "m-", lw=1.8, label=f"UCB (next x={X_grid[np.argmax(ucb_vals),0]:.2f})")
axes[1].plot(X_grid, mu / (mu.max() + 1e-9), "b:", lw=1.5, label=f"Greedy (next x={X_grid[np.argmax(mu),0]:.2f})")
axes[1].set_xlabel("Parameter x (e.g. Normalized Temperature)")
axes[1].set_ylabel("Normalized Score")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

📘 **看下图的对比：**
- **Greedy（蓝色虚线）**：死盯着左边的已知小山头（$x \approx 2.3$），完全看不到右边 $x=7.8$ 的真正高峰；
- **EI（绿线）和 UCB（紫线）**：在右边空白区域 $x > 4$ 给出了最高分，果断决定跳出局部最优，去探索未知空间！

---
## 3. 1D 材料工艺寻优：直观观察 6 轮迭代全过程

▶️ 运行下面的代码，逐轮可视化贝叶斯优化（EI）从 3 个初始点开始，如何只用 **6 次新实验** 就精准锁定 $x=7.8$ 的全局最优点。

In [ ]:
X_obs = X_init.copy()
y_obs = y_init.copy()

fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), sharex=True, sharey=True)
for step, ax in enumerate(axes.ravel(), start=1):
    gp_step = GaussianProcessRegressor(kernel=C(1.0) * RBF(1.0), alpha=1e-5, random_state=0)
    gp_step.fit(X_obs, y_obs)
    mu, std = gp_step.predict(X_grid, return_std=True)
    ei = acq_ei(mu, std, y_obs.max(), xi=0.01)
    next_x = X_grid[np.argmax(ei)]
    next_y = hidden_process(next_x).item()

    ax.plot(X_grid, hidden_process(X_grid), "k--", alpha=0.4)
    ax.plot(X_grid, mu, "b-", lw=1.5)
    ax.fill_between(X_grid.ravel(), mu - 1.96 * std, mu + 1.96 * std, color="blue", alpha=0.15)
    ax.scatter(X_obs, y_obs, c="red", s=35, zorder=5)
    ax.scatter([next_x[0]], [next_y], c="lime", edgecolors="black", s=80, marker="*", zorder=6, label=f"Next x={next_x[0]:.2f}")
    ax.set_title(f"Step {step} | Best y={max(y_obs.max(), next_y):.3f}")
    ax.legend(loc="upper left", fontsize=8)
    ax.grid(alpha=0.25)

    X_obs = np.vstack([X_obs, next_x.reshape(1, 1)])
    y_obs = np.append(y_obs, next_y)

plt.suptitle("Bayesian Optimization (EI) Escaping Local Optimum in 6 Steps", fontsize=12)
plt.tight_layout()
plt.show()

---
## 4. 真实材料化学实战：Buchwald-Hartwig 钯催化偶联反应优化

📘 现实中的材料与有机合成筛选往往不是一维连续温度，而是**离散的组合空间**：手头有若干种配体（Ligand）、碱（Base）、添加剂（Additive）和芳基卤（Aryl Halide），一共能组合出几千个反应。

`data/bh-reactions.csv` 是 *Science* 2018 经典高通量实验数据集（Ahneman et al.）：包含 **3955 个钯催化 C-N 偶联反应** 及其真实实验产率 `yield`（0% ~ 100%）。
我们把每个反应的 4 个核心组分（`ligand`, `additive`, `base`, `aryl halide`）做 **One-Hot 编码**（或者分子指纹编码），在上面模拟一次**预算只有 25 次实验**的贝叶斯优化，看看它比"随机盲试"快多少！

🤔 **先猜一猜：** 3955 个反应里，产率最高的能达到 99.9%，但平均产率只有 33% 左右。如果只做 25 次实验（不到总空间的 0.7%），随机抽样和贝叶斯优化分别能找到多高产率的反应？

▶️ 运行下面的代码，在 CPU 上只需几秒即可完成对比（我们跑 5 个不同的随机起点，看平均曲线）。

In [ ]:
bh = pd.read_csv(find_file("bh-reactions.csv"))
print(f"反应总数: {len(bh)}, 平均产率: {bh['yield'].mean():.1f}%, 最高产率: {bh['yield'].max():.1f}%")

# 用 4 个反应组分的 One-Hot 编码作为特征（共 42 维，简单高效）
X_oh = pd.get_dummies(bh[["ligand", "additive", "base", "aryl halide"]], dtype=float).values
y_yield = bh["yield"].values
print("反应组合特征矩阵形状:", X_oh.shape)

def run_bo_reaction(X, y, n_init=5, n_steps=20, seed=0, strategy="ei"):
    rng = np.random.RandomState(seed)
    n_total = len(y)
    observed = list(rng.choice(n_total, size=n_init, replace=False))
    best_history = [y[observed].max()]

    for _ in range(n_steps):
        if strategy == "random":
            unseen = [i for i in range(n_total) if i not in observed]
            next_idx = rng.choice(unseen)
        else:
            # 用 RBF + WhiteKernel 在已观测反应上拟合 GPR
            kernel = C(1.0) * RBF(length_scale=2.0) + WhiteKernel(noise_level=1.0)
            gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, random_state=seed)
            gp.fit(X[observed], y[observed])

            unseen_idx = np.array([i for i in range(n_total) if i not in observed])
            mu, std = gp.predict(X[unseen_idx], return_std=True)
            if strategy == "ei":
                scores = acq_ei(mu, std, y[observed].max(), xi=0.5)
            elif strategy == "ucb":
                scores = acq_ucb(mu, std, kappa=2.0)
            next_idx = unseen_idx[np.argmax(scores)]

        observed.append(next_idx)
        best_history.append(y[observed].max())
    return np.array(best_history)

# 跑 5 个随机种子（每次 5 个初始实验 + 20 轮 BO = 25 次实验）
seeds = [0, 1, 2, 3, 4]
curves_rand = np.array([run_bo_reaction(X_oh, y_yield, seed=s, strategy="random") for s in seeds])
curves_ei   = np.array([run_bo_reaction(X_oh, y_yield, seed=s, strategy="ei") for s in seeds])
curves_ucb  = np.array([run_bo_reaction(X_oh, y_yield, seed=s, strategy="ucb") for s in seeds])

steps = np.arange(5, 26)
plt.figure(figsize=(8.5, 4.8))
for name, mat, col in [("Random Search", curves_rand, "gray"),
                       ("BO (EI)", curves_ei, "green"),
                       ("BO (UCB, k=2)", curves_ucb, "blue")]:
    mean_c = mat.mean(axis=0)
    std_c = mat.std(axis=0)
    plt.plot(steps, mean_c, lw=2.2, color=col, label=f"{name} (Final={mean_c[-1]:.1f}%)")
    plt.fill_between(steps, mean_c - std_c, mean_c + std_c, color=col, alpha=0.12)

plt.axhline(y_yield.max(), color="red", linestyle="--", lw=1, label=f"Global Max ({y_yield.max():.1f}%)")
plt.xlabel("Total Number of Experiments")
plt.ylabel("Best Reaction Yield Found (%)")
plt.title("Buchwald-Hartwig Coupling Optimization (3955 Candidates)")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

📘 **结果非常震撼：** 在 3955 个可能配方里，**仅仅做了 25 次实验（0.6%）**，贝叶斯优化（EI/UCB）就已经稳定找到了 **94%~97% 以上**的高产率反应配方，远超随机盲试！

---
## 5. 贝叶斯优化的边界与使用忠告

💡 贝叶斯优化不是万能的，实际使用时要注意三条边界：
1. **维数灾难**：经典 GPR 最适合 **2~20 维** 的低维参数优化（温度、压强、浓度、配比、离散组合的 One-Hot）。如果特征超过几百维，距离会失效，需要降维或改用随机森林代理模型（如 SMAC）或潜空间优化（见 `09a`）。
2. **算力瓶颈**：标准 GPR 求逆矩阵的复杂度是 $O(N^3)$。当实验数量超过 $2000$ 条时，每次拟合都会变慢——不过到了几千条数据时，往往就已经该切回深度学习或主动学习集成模型了。
3. **批量实验（Batch BO）**：现实中实验室的平行反应釜一次能跑 8 个或 24 个孔板，而上面写的采集函数每次只选 1 个点。工业界会用"克里金假想观测（Kriging Believer）"或 `qEI` 一次性选出互不重复的 8 个推荐配方。

---
## 🏋️ 练习

### 练习 04b-1：调节 UCB 的探索系数 $\kappa$
在第 3 节的 1D 双峰工艺曲线上，把采集函数从 `EI` 换成 `UCB`，分别测试 $\kappa = 0.1$（极度保守利用）、$\kappa = 2.0$（平衡）和 $\kappa = 10.0$（极度探索），运行 6 步迭代后，谁最先找到右边的全局高峰 $x \approx 7.8$？谁困在了左边？

In [ ]:
# 在这里写你的代码

### 练习 04b-2：二维材料工艺参数寻优（温度 + 浓度）
构建一个二维材料目标函数 $f(T, c)$（例如 $T \in [0, 10], c \in [0, 10]$，包含一个全局峰值和一个次优峰值），在 $25 \times 25 = 625$ 的网格上用 `GaussianProcessRegressor` + `EI` 跑 15 轮贝叶斯优化，对比随机抽样 15 次找到的最大值，并把最终选中的点画在二维等高线图上。

In [ ]:
# 在这里写你的代码

### 练习 04b-3：如果改用分子指纹代替 One-Hot 编码？
在第 4 节的 Buchwald-Hartwig 数据集上，我们用了各组分的 One-Hot 编码（模型完全不知道两个配体长什么样，只知道它们是不是同一个编号）。
试着对 `ligand`、`additive`、`base`、`aryl halide` 分别算 256 维的 Morgan 指纹并拼接起来（用 `PCA` 降到 30 维以保证 GPR 速度），再跑一次 5 个种子的 `EI` 寻优，对比纯 One-Hot 编码的效果——**化学结构特征有没有帮助模型在没见过的配体间泛化？**

In [ ]:
# 在这里写你的代码